<a href="https://colab.research.google.com/github/Ozzyimandias218/Codemania/blob/main/Study_Day_Planner.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [19]:
%pip install -U streamlit langchain-groq groq langchain-core langgraph requests python-dotenv

In [20]:
import os
import getpass

if not os.environ.get("GROQ_API_KEY"):
    os.environ["GROQ_API_KEY"] = getpass.getpass("Enter your GROQ_API_KEY: ")

In [21]:
import os
from google.colab import userdata

# Try to get the key from Google Colab Secrets (recommended)
try:
    os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
    print("GROQ_API_KEY successfully loaded from Colab Secrets.")
except Exception:
    # Fallback to manual prompt if not set in Secrets
    import getpass
    os.environ["GROQ_API_KEY"] = getpass.getpass("Enter your GROQ_API_KEY: ")
    print("GROQ_API_KEY set for this session.")

GROQ_API_KEY successfully loaded from Colab Secrets.


In [22]:
# --- IMPORTS ---
import os
from typing import Annotated, TypedDict

import requests
from langchain_core.messages import AIMessage, SystemMessage
from langchain_core.tools import tool
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import START, StateGraph
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.types import Command, interrupt

# --- TOOLS ---
TASKS: list[str] = []  # temporary task list (lost when the program restarts)


@tool
def add_task(title: str) -> str:
    """Add a new study task to the user's task list."""
    TASKS.append(title)
    return f"Task added: {title}"


@tool
def list_tasks() -> str:
    """Show all tasks in the user's task list."""
    if not TASKS:
        return "The task list is empty."
    return "\n".join(f"{i + 1}. {t}" for i, t in enumerate(TASKS))


@tool
def delete_task(title: str) -> str:
    """Delete a task from the list. Always asks the human for approval first."""
    wanted = title.strip().lower()
    matches = [t for t in TASKS if t.lower() == wanted] or [
        t for t in TASKS if wanted in t.lower()
    ]
    if not matches:
        return f"Task not found: {title}"
    target = matches[0]

    # The graph pauses here until the human answers (Command(resume=True/False)).
    approved = interrupt(
        {
            "action": "delete_task",
            "task": target,
            "question": f"Do you really want to delete the task '{target}'?",
        }
    )
    if approved is True:
        TASKS.remove(target)
        return f"Task deleted: {target}"
    return f"Deletion cancelled. The task '{target}' was kept."


@tool
def get_weather(city: str) -> str:
    """Get the current temperature, wind and rain for a city using Open-Meteo."""
    try:
        geo = requests.get(
            "https://geocoding-api.open-meteo.com/v1/search",
            params={"name": city, "count": 1},
            timeout=10,
        ).json()
        if not geo.get("results"):
            return f"City not found: {city}"
        place = geo["results"][0]
        now = requests.get(
            "https://api.open-meteo.com/v1/forecast",
            params={
                "latitude": place["latitude"],
                "longitude": place["longitude"],
                "current": "temperature_2m,wind_speed_10m,precipitation",
            },
            timeout=10,
        ).json()["current"]
        return (
            f"{place['name']}: {now['temperature_2m']}°C, "
            f"wind {now['wind_speed_10m']} km/h, "
            f"precipitation {now['precipitation']} mm"
        )
    except Exception as e:
        return f"Weather service error: {e}"


@tool
def search_books(topic: str) -> str:
    """Search books by topic using Open Library and return a few titles with authors."""
    try:
        data = requests.get(
            "https://openlibrary.org/search.json",
            params={"q": topic, "limit": 5, "fields": "title,author_name"},
            timeout=10,
        ).json()
        docs = data.get("docs", [])
        if not docs:
            return f"No books found for: {topic}"
        return "\n".join(
            f"- {d.get('title', '?')} by {', '.join(d.get('author_name', ['unknown'])[:2])}"
            for d in docs
        )
    except Exception as e:
        return f"Book service error: {e}"


tools = [add_task, list_tasks, delete_task, get_weather, search_books]

# --- GRAPH ---
SYSTEM_PROMPT = """You are Study Day Planner, a friendly assistant for university students.
You help the user manage study tasks, check the weather and find books.
Rules:
- Use add_task, list_tasks and delete_task for the task list. Never pretend a task was changed without calling the tool.
- Use get_weather for weather questions. Use search_books for book requests (English topic words work best).
- Personal behavior: when the user asks where to study or plans a study day, call get_weather first,
  then suggest a study place based on it (rain, cold or strong wind -> library or cafe indoors;
  mild and dry -> a park or an outdoor terrace), and also show books on the user's topic.
- Delete only ONE task per tool call turn; if the user wants several deleted, handle them one after another.
- Keep answers short. Reply in the language the user writes in."""


class State(TypedDict):
    # add_messages APPENDS new messages to the history instead of replacing it
    messages: Annotated[list, add_messages]


def build_graph(llm=None):
    if llm is None:
        try:
            from langchain_groq import ChatGroq

            llm = ChatGroq(
                model=os.environ.get("GROQ_MODEL", "llama-3.3-70b-versatile"),
                temperature=0,
            )
        except ImportError as e:
            raise ImportError(
                "Groq packages are missing or out of date. Run:\n"
                "  %pip install -U langchain-groq groq langchain-core langgraph requests\n"
                "then RESTART the kernel/session and run again.\n"
                f"Original error: {e}"
            ) from e
    llm_with_tools = llm.bind_tools(tools)

    def assistant(state: State):
        error = None
        for _ in range(3):  # Groq sometimes rejects a malformed tool call -> retry
            try:
                reply = llm_with_tools.invoke(
                    [SystemMessage(content=SYSTEM_PROMPT)] + state["messages"]
                )
                return {"messages": [reply]}
            except Exception as e:  # noqa: BLE001
                error = e
        return {
            "messages": [
                AIMessage(
                    content=f"Sorry, the language model failed ({type(error).__name__}: "
                    f"{error}). Please rephrase your request and try again."
                )
            ]
        }

    builder = StateGraph(State)
    builder.add_node("assistant", assistant)
    builder.add_node("tools", ToolNode(tools))
    builder.add_edge(START, "assistant")
    # model asked for a tool -> "tools"; final answer -> END
    builder.add_conditional_edges("assistant", tools_condition)
    builder.add_edge("tools", "assistant")  # tool result goes back to the assistant
    return builder.compile(checkpointer=InMemorySaver())


def get_interrupt(graph, config):
    """Return the pending approval request (dict) or None, read from the graph state."""
    for task in graph.get_state(config).tasks:
        if task.interrupts:
            return task.interrupts[0].value
    return None


# --- HELPERS ---
def last_text(result) -> str:
    content = result["messages"][-1].content
    if isinstance(content, list):  # some models return a list of content blocks
        return " ".join(b.get("text", "") if isinstance(b, dict) else str(b) for b in content)
    return content


def chat(graph, text: str, config: dict):
    """Send one user message. Returns (result, pending_approval_or_None)."""
    result = graph.invoke({"messages": [("user", text)]}, config)
    return result, get_interrupt(graph, config)


def answer(graph, approved: bool, config: dict):
    """Resume a paused graph with the human's decision."""
    result = graph.invoke(Command(resume=approved), config)
    return result, get_interrupt(graph, config)

In [23]:
%%writefile agent.py
# --- IMPORTS ---
import os
from typing import Annotated, TypedDict

import requests
from langchain_core.messages import AIMessage, SystemMessage
from langchain_core.tools import tool
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import START, StateGraph
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.types import Command, interrupt

# --- TOOLS ---
TASKS: list[str] = []  # temporary task list (lost when the program restarts)


@tool
def add_task(title: str) -> str:
    """Add a new study task to the user's task list."""
    TASKS.append(title)
    return f"Task added: {title}"


@tool
def list_tasks() -> str:
    """Show all tasks in the user's task list."""
    if not TASKS:
        return "The task list is empty."
    return "\n".join(f"{i + 1}. {t}" for i, t in enumerate(TASKS))


@tool
def delete_task(title: str) -> str:
    """Delete a task from the list. Always asks the human for approval first."""
    wanted = title.strip().lower()
    matches = [t for t in TASKS if t.lower() == wanted] or [
        t for t in TASKS if wanted in t.lower()
    ]
    if not matches:
        return f"Task not found: {title}"
    target = matches[0]

    # The graph pauses here until the human answers (Command(resume=True/False)).
    approved = interrupt(
        {
            "action": "delete_task",
            "task": target,
            "question": f"Do you really want to delete the task '{target}'?",
        }
    )
    if approved is True:
        TASKS.remove(target)
        return f"Task deleted: {target}"
    return f"Deletion cancelled. The task '{target}' was kept."


@tool
def get_weather(city: str) -> str:
    """Get the current temperature, wind and rain for a city using Open-Meteo."""
    try:
        geo = requests.get(
            "https://geocoding-api.open-meteo.com/v1/search",
            params={"name": city, "count": 1},
            timeout=10,
        ).json()
        if not geo.get("results"):
            return f"City not found: {city}"
        place = geo["results"][0]
        now = requests.get(
            "https://api.open-meteo.com/v1/forecast",
            params={
                "latitude": place["latitude"],
                "longitude": place["longitude"],
                "current": "temperature_2m,wind_speed_10m,precipitation",
            },
            timeout=10,
        ).json()["current"]
        return (
            f"{place['name']}: {now['temperature_2m']}°C, "
            f"wind {now['wind_speed_10m']} km/h, "
            f"precipitation {now['precipitation']} mm"
        )
    except Exception as e:
        return f"Weather service error: {e}"


@tool
def search_books(topic: str) -> str:
    """Search books by topic using Open Library and return a few titles with authors."""
    try:
        data = requests.get(
            "https://openlibrary.org/search.json",
            params={"q": topic, "limit": 5, "fields": "title,author_name"},
            timeout=10,
        ).json()
        docs = data.get("docs", [])
        if not docs:
            return f"No books found for: {topic}"
        return "\n".join(
            f"- {d.get('title', '?')} by {', '.join(d.get('author_name', ['unknown'])[:2])}"
            for d in docs
        )
    except Exception as e:
        return f"Book service error: {e}"


tools = [add_task, list_tasks, delete_task, get_weather, search_books]

# --- GRAPH ---
SYSTEM_PROMPT = """You are Study Day Planner, a friendly assistant for university students.
You help the user manage study tasks, check the weather and find books.
Rules:
- Use add_task, list_tasks and delete_task for the task list. Never pretend a task was changed without calling the tool.
- Use get_weather for weather questions. Use search_books for book requests (English topic words work best).
- Personal behavior: when the user asks where to study or plans a study day, call get_weather first,
  then suggest a study place based on it (rain, cold or strong wind -> library or cafe indoors;
  mild and dry -> a park or an outdoor terrace), and also show books on the user's topic.
- Delete only ONE task per tool call turn; if the user wants several deleted, handle them one after another.
- Keep answers short. Reply in the language the user writes in."""


class State(TypedDict):
    messages: Annotated[list, add_messages]


def build_graph(llm=None):
    if llm is None:
        try:
            from langchain_groq import ChatGroq

            llm = ChatGroq(
                model=os.environ.get("GROQ_MODEL", "llama-3.3-70b-versatile"),
                temperature=0,
            )
        except ImportError as e:
            raise ImportError(
                "Groq packages are missing or out of date. Run:\n"
                "  %pip install -U langchain-groq groq langchain-core langgraph requests\n"
                "then RESTART the kernel/session and run again.\n"
                f"Original error: {e}"
            ) from e
    llm_with_tools = llm.bind_tools(tools)

    def assistant(state: State):
        error = None
        for _ in range(3):  # Groq sometimes rejects a malformed tool call -> retry
            try:
                reply = llm_with_tools.invoke(
                    [SystemMessage(content=SYSTEM_PROMPT)] + state["messages"]
                )
                return {"messages": [reply]}
            except Exception as e:  # noqa: BLE001
                error = e
        return {
            "messages": [
                AIMessage(
                    content=f"Sorry, the language model failed ({type(error).__name__}: "
                    f"{error}). Please rephrase your request and try again."
                )
            ]
        }

    builder = StateGraph(State)
    builder.add_node("assistant", assistant)
    builder.add_node("tools", ToolNode(tools))
    builder.add_edge(START, "assistant")
    builder.add_conditional_edges("assistant", tools_condition)
    builder.add_edge("tools", "assistant")
    return builder.compile(checkpointer=InMemorySaver())


def get_interrupt(graph, config):
    for task in graph.get_state(config).tasks:
        if task.interrupts:
            return task.interrupts[0].value
    return None


# --- HELPERS ---
def last_text(result) -> str:
    content = result["messages"][-1].content
    if isinstance(content, list):
        return " ".join(b.get("text", "") if isinstance(b, dict) else str(b) for b in content)
    return content


def chat(graph, text: str, config: dict):
    result = graph.invoke({"messages": [("user", text)]}, config)
    return result, get_interrupt(graph, config)


def answer(graph, approved: bool, config: dict):
    result = graph.invoke(Command(resume=approved), config)
    return result, get_interrupt(graph, config)

Overwriting agent.py


In [24]:
%%writefile agent.py
# --- IMPORTS ---
import os
import json
from typing import Annotated, TypedDict

import requests
from langchain_core.messages import AIMessage, SystemMessage
from langchain_core.tools import tool
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import START, StateGraph
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.types import Command, interrupt

# --- PERSISTENCE HELPERS ---
TASKS_FILE = "tasks.json"

def load_tasks() -> list[str]:
    if os.path.exists(TASKS_FILE):
        try:
            with open(TASKS_FILE, "r") as f:
                return json.load(f)
        except Exception:
            return []
    return []

def save_tasks(tasks: list[str]):
    try:
        with open(TASKS_FILE, "w") as f:
            json.dump(tasks, f)
    except Exception:
        pass

# Initialize TASKS list
TASKS: list[str] = load_tasks()

# --- TOOLS ---
@tool
def add_task(title: str) -> str:
    """Add a new study task to the user's task list."""
    global TASKS
    TASKS = load_tasks()
    TASKS.append(title)
    save_tasks(TASKS)
    return f"Task added and saved to file: {title}"

@tool
def list_tasks() -> str:
    """Show all tasks in the user's task list."""
    global TASKS
    TASKS = load_tasks()
    if not TASKS:
        return "The task list is empty."
    return "\n".join(f"{i + 1}. {t}" for i, t in enumerate(TASKS))

@tool
def delete_task(title: str) -> str:
    """Delete a task from the list. Always asks the human for approval first."""
    global TASKS
    TASKS = load_tasks()
    wanted = title.strip().lower()
    matches = [t for t in TASKS if t.lower() == wanted] or [
        t for t in TASKS if wanted in t.lower()
    ]
    if not matches:
        return f"Task not found: {title}"
    target = matches[0]

    # The graph pauses here until the human answers (Command(resume=True/False)).
    approved = interrupt(
        {
            "action": "delete_task",
            "task": target,
            "question": f"Do you really want to delete the task '{target}'?",
        }
    )
    if approved is True:
        TASKS = load_tasks()
        if target in TASKS:
            TASKS.remove(target)
            save_tasks(TASKS)
        return f"Task deleted from file: {target}"
    return f"Deletion cancelled. The task '{target}' was kept."

@tool
def get_weather(city: str) -> str:
    """Get the current temperature, wind and rain for a city using Open-Meteo."""
    try:
        geo = requests.get(
            "https://geocoding-api.open-meteo.com/v1/search",
            params={"name": city, "count": 1},
            timeout=10,
        ).json()
        if not geo.get("results"):
            return f"City not found: {city}"
        place = geo["results"][0]
        now = requests.get(
            "https://api.open-meteo.com/v1/forecast",
            params={
                "latitude": place["latitude"],
                "longitude": place["longitude"],
                "current": "temperature_2m,wind_speed_10m,precipitation",
            },
            timeout=10,
        ).json()["current"]
        return (
            f"{place['name']}: {now['temperature_2m']}°C, "
            f"wind {now['wind_speed_10m']} km/h, "
            f"precipitation {now['precipitation']} mm"
        )
    except Exception as e:
        return f"Weather service error: {e}"

@tool
def search_books(topic: str) -> str:
    """Search books by topic using Open Library and return a few titles with authors."""
    try:
        data = requests.get(
            "https://openlibrary.org/search.json",
            params={"q": topic, "limit": 5, "fields": "title,author_name"},
            timeout=10,
        ).json()
        docs = data.get("docs", [])
        if not docs:
            return f"No books found for: {topic}"
        return "\n".join(
            f"- {d.get('title', '?')} by {', '.join(d.get('author_name', ['unknown'])[:2])}"
            for d in docs
        )
    except Exception as e:
        return f"Book service error: {e}"

tools = [add_task, list_tasks, delete_task, get_weather, search_books]

# --- GRAPH ---
SYSTEM_PROMPT = """You are Study Day Planner, a friendly assistant for university students.
You help the user manage study tasks, check the weather and find books.
Rules:
- Use add_task, list_tasks and delete_task for the task list. Never pretend a task was changed without calling the tool.
- Use get_weather for weather questions. Use search_books for book requests (English topic words work best).
- Personal behavior: when the user asks where to study or plans a study day, call get_weather first,
  then suggest a study place based on it (rain, cold or strong wind -> library or cafe indoors;
  mild and dry -> a park or an outdoor terrace), and also show books on the user's topic.
- Delete only ONE task per tool call turn; if the user wants several deleted, handle them one after another.
- Keep answers short. Reply in the language the user writes in."""

class State(TypedDict):
    messages: Annotated[list, add_messages]

def build_graph(llm=None):
    if llm is None:
        try:
            from langchain_groq import ChatGroq

            llm = ChatGroq(
                model=os.environ.get("GROQ_MODEL", "llama-3.3-70b-versatile"),
                temperature=0,
            )
        except ImportError as e:
            raise ImportError(
                "Groq packages are missing or out of date. Run:\n"
                "  %pip install -U langchain-groq groq langchain-core langgraph requests\n"
                "then RESTART the kernel/session and run again.\n"
                f"Original error: {e}"
            ) from e
    llm_with_tools = llm.bind_tools(tools)

    def assistant(state: State):
        error = None
        for _ in range(3):  # Groq sometimes rejects a malformed tool call -> retry
            try:
                reply = llm_with_tools.invoke(
                    [SystemMessage(content=SYSTEM_PROMPT)] + state["messages"]
                )
                return {"messages": [reply]}
            except Exception as e:  # noqa: BLE001
                error = e
        return {
            "messages": [
                AIMessage(
                    content=f"Sorry, the language model failed ({type(error).__name__}: "
                    f"{error}). Please rephrase your request and try again."
                )
            ]
        }

    builder = StateGraph(State)
    builder.add_node("assistant", assistant)
    builder.add_node("tools", ToolNode(tools))
    builder.add_edge(START, "assistant")
    builder.add_conditional_edges("assistant", tools_condition)
    builder.add_edge("tools", "assistant")
    return builder.compile(checkpointer=InMemorySaver())

def get_interrupt(graph, config):
    for task in graph.get_state(config).tasks:
        if task.interrupts:
            return task.interrupts[0].value
    return None

# --- HELPERS ---
def last_text(result) -> str:
    content = result["messages"][-1].content
    if isinstance(content, list):
        return " ".join(b.get("text", "") if isinstance(b, dict) else str(b) for b in content)
    return content

def chat(graph, text: str, config: dict):
    result = graph.invoke({"messages": [("user", text)]}, config)
    return result, get_interrupt(graph, config)

def answer(graph, approved: bool, config: dict):
    result = graph.invoke(Command(resume=approved), config)
    return result, get_interrupt(graph, config)


Overwriting agent.py


In [25]:
%%writefile app.py
import os
import uuid
import streamlit as st
try:
    from dotenv import load_dotenv
    load_dotenv()
except Exception:
    pass

st.set_page_config(page_title="Study Day Planner", page_icon="📚")
st.title("📚 Study Day Planner")
st.caption("Tasks, weather and books — with approval before deleting anything.")

if not os.environ.get("GROQ_API_KEY"):
    st.error("GROQ_API_KEY is not set. Please set it in your environment or secrets.")
    st.stop()

from agent import answer, build_graph, chat, last_text

@st.cache_resource
def get_graph():
    return build_graph()

graph = get_graph()
if "thread_id" not in st.session_state:
    st.session_state.thread_id = str(uuid.uuid4())
    st.session_state.history = []
    st.session_state.pending = None
config = {"configurable": {"thread_id": st.session_state.thread_id}}

for role, text in st.session_state.history:
    st.chat_message(role).write(text)

def finish(result, pending):
    st.session_state.pending = pending
    if not pending:
        st.session_state.history.append(("assistant", last_text(result)))

if st.session_state.pending:
    st.warning(st.session_state.pending["question"])
    c1, c2 = st.columns(2)
    decision = None
    if c1.button("✅ Yes, delete"):
        decision = True
    if c2.button("❌ No, keep it"):
        decision = False
    if decision is not None:
        finish(*answer(graph, decision, config))
        st.rerun()
else:
    if prompt := st.chat_input("Ask me about tasks, weather or books…"):
        st.session_state.history.append(("user", prompt))
        finish(*chat(graph, prompt, config))
        st.rerun()

Overwriting app.py


In [26]:
"""Streamlit chat UI: streamlit run app.py"""
import os
import sys
import uuid

# Ensure the current directory is in the Python path so local modules like 'agent' can be loaded
sys.path.append(os.path.abspath('.'))

import streamlit as st
try:
    from dotenv import load_dotenv

    load_dotenv()
except Exception:  # skip .env if python-dotenv is missing or fails
    pass
st.set_page_config(page_title="Study Day Planner", page_icon="📚")
st.title("📚 Study Day Planner")
st.caption("Tasks, weather and books — with approval before deleting anything.")

if not os.environ.get("GROQ_API_KEY"):
    st.error("GROQ_API_KEY is not set. Copy .env.example to .env and add your key.")
    st.stop()

from agent import answer, build_graph, chat, last_text


@st.cache_resource
def get_graph():
    return build_graph()


graph = get_graph()
if "thread_id" not in st.session_state:
    st.session_state.thread_id = str(uuid.uuid4())
    st.session_state.history = []   # [(role, text)]
    st.session_state.pending = None  # approval request, if any
config = {"configurable": {"thread_id": st.session_state.thread_id}}

for role, text in st.session_state.history:
    st.chat_message(role).write(text)


def finish(result, pending):
    st.session_state.pending = pending
    if not pending:
        st.session_state.history.append(("assistant", last_text(result)))


if st.session_state.pending:
    st.warning(st.session_state.pending["question"])
    c1, c2 = st.columns(2)
    decision = None
    if c1.button("✅ Yes, delete"):
        decision = True
    if c2.button("❌ No, keep it"):
        decision = False
    if decision is not None:
        finish(*answer(graph, decision, config))
        st.rerun()
else:
    if prompt := st.chat_input("Ask me about tasks, weather or books…"):
        st.session_state.history.append(("user", prompt))
        finish(*chat(graph, prompt, config))
        st.rerun()

2026-11-22 21:02:34.388 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-22 21:02:34.394 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-22 21:02:34.397 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-22 21:02:34.404 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-22 21:02:34.409 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-22 21:02:34.416 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-22 21:02:34.418 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-22 21:02:34.426 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar

In [27]:
import urllib.request
# Get your public IP address (you will need to enter this in localtunnel to access the app)
print("Your Tunnel Password (Public IP):", urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode('utf8').strip())

Your Tunnel Password (Public IP): 34.185.82.124


In [ ]:
# Run Streamlit in the background and expose it using localtunnel
get_ipython().system('nohup streamlit run app.py --server.port 8501 &')
get_ipython().system('npx localtunnel --port 8501')

nohup: appending output to 'nohup.out'
⠙⠹⠸⠼⠴⠦⠧

In [ ]:
# --- IMPORTS ---
import os
from typing import Annotated, TypedDict

import requests
from langchain_core.messages import AIMessage, SystemMessage
from langchain_core.tools import tool
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import START, StateGraph
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.types import Command, interrupt

# --- TOOLS ---
TASKS: list[str] = []  # temporary task list (lost when the program restarts)


@tool
def add_task(title: str) -> str:
    """Add a new study task to the user's task list."""
    TASKS.append(title)
    return f"Task added: {title}"


@tool
def list_tasks() -> str:
    """Show all tasks in the user's task list."""
    if not TASKS:
        return "The task list is empty."
    return "\n".join(f"{i + 1}. {t}" for i, t in enumerate(TASKS))


@tool
def delete_task(title: str) -> str:
    """Delete a task from the list. Always asks the human for approval first."""
    wanted = title.strip().lower()
    matches = [t for t in TASKS if t.lower() == wanted] or [
        t for t in TASKS if wanted in t.lower()
    ]
    if not matches:
        return f"Task not found: {title}"
    target = matches[0]

    # The graph pauses here until the human answers (Command(resume=True/False)).
    approved = interrupt(
        {
            "action": "delete_task",
            "task": target,
            "question": f"Do you really want to delete the task '{target}'?",
        }
    )
    if approved is True:
        TASKS.remove(target)
        return f"Task deleted: {target}"
    return f"Deletion cancelled. The task '{target}' was kept."


@tool
def get_weather(city: str) -> str:
    """Get the current temperature, wind and rain for a city using Open-Meteo."""
    try:
        geo = requests.get(
            "https://geocoding-api.open-meteo.com/v1/search",
            params={"name": city, "count": 1},
            timeout=10,
        ).json()
        if not geo.get("results"):
            return f"City not found: {city}"
        place = geo["results"][0]
        now = requests.get(
            "https://api.open-meteo.com/v1/forecast",
            params={
                "latitude": place["latitude"],
                "longitude": place["longitude"],
                "current": "temperature_2m,wind_speed_10m,precipitation",
            },
            timeout=10,
        ).json()["current"]
        return (
            f"{place['name']}: {now['temperature_2m']}°C, "
            f"wind {now['wind_speed_10m']} km/h, "
            f"precipitation {now['precipitation']} mm"
        )
    except Exception as e:
        return f"Weather service error: {e}"


@tool
def search_books(topic: str) -> str:
    """Search books by topic using Open Library and return a few titles with authors."""
    try:
        data = requests.get(
            "https://openlibrary.org/search.json",
            params={"q": topic, "limit": 5, "fields": "title,author_name"},
            timeout=10,
        ).json()
        docs = data.get("docs", [])
        if not docs:
            return f"No books found for: {topic}"
        return "\n".join(
            f"- {d.get('title', '?')} by {', '.join(d.get('author_name', ['unknown'])[:2])}"
            for d in docs
        )
    except Exception as e:
        return f"Book service error: {e}"


tools = [add_task, list_tasks, delete_task, get_weather, search_books]

# --- GRAPH ---
SYSTEM_PROMPT = """You are Study Day Planner, a friendly assistant for university students.
You help the user manage study tasks, check the weather and find books.
Rules:
- Use add_task, list_tasks and delete_task for the task list. Never pretend a task was changed without calling the tool.
- Use get_weather for weather questions. Use search_books for book requests (English topic words work best).
- Personal behavior: when the user asks where to study or plans a study day, call get_weather first,
  then suggest a study place based on it (rain, cold or strong wind -> library or cafe indoors;
  mild and dry -> a park or an outdoor terrace), and also show books on the user's topic.
- Delete only ONE task per tool call turn; if the user wants several deleted, handle them one after another.
- Keep answers short. Reply in the language the user writes in."""


class State(TypedDict):
    # add_messages APPENDS new messages to the history instead of replacing it
    messages: Annotated[list, add_messages]


def build_graph(llm=None):
    if llm is None:
        try:
            from langchain_groq import ChatGroq

            llm = ChatGroq(
                model=os.environ.get("GROQ_MODEL", "llama-3.3-70b-versatile"),
                temperature=0,
            )
        except ImportError as e:
            raise ImportError(
                "Groq packages are missing or out of date. Run:\n"
                "  %pip install -U langchain-groq groq langchain-core langgraph requests\n"
                "then RESTART the kernel/session and run again.\n"
                f"Original error: {e}"
            ) from e
    llm_with_tools = llm.bind_tools(tools)

    def assistant(state: State):
        error = None
        for _ in range(3):  # Groq sometimes rejects a malformed tool call -> retry
            try:
                reply = llm_with_tools.invoke(
                    [SystemMessage(content=SYSTEM_PROMPT)] + state["messages"]
                )
                return {"messages": [reply]}
            except Exception as e:  # noqa: BLE001
                error = e
        return {
            "messages": [
                AIMessage(
                    content=f"Sorry, the language model failed ({type(error).__name__}: "
                    f"{error}). Please rephrase your request and try again."
                )
            ]
        }

    builder = StateGraph(State)
    builder.add_node("assistant", assistant)
    builder.add_node("tools", ToolNode(tools))
    builder.add_edge(START, "assistant")
    # model asked for a tool -> "tools"; final answer -> END
    builder.add_conditional_edges("assistant", tools_condition)
    builder.add_edge("tools", "assistant")  # tool result goes back to the assistant
    return builder.compile(checkpointer=InMemorySaver())


def get_interrupt(graph, config):
    """Return the pending approval request (dict) or None, read from the graph state."""
    for task in graph.get_state(config).tasks:
        if task.interrupts:
            return task.interrupts[0].value
    return None


# --- HELPERS ---
def last_text(result) -> str:
    content = result["messages"][-1].content
    if isinstance(content, list):  # some models return a list of content blocks
        return " ".join(b.get("text", "") if isinstance(b, dict) else str(b) for b in content)
    return content


def chat(graph, text: str, config: dict):
    """Send one user message. Returns (result, pending_approval_or_None)."""
    result = graph.invoke({"messages": [("user", text)]}, config)
    return result, get_interrupt(graph, config)


def answer(graph, approved: bool, config: dict):
    """Resume a paused graph with the human's decision."""
    result = graph.invoke(Command(resume=approved), config)
    return result, get_interrupt(graph, config)

In [ ]:
"""Streamlit chat UI: streamlit run app.py"""
import os
import uuid

import streamlit as st
try:
    from dotenv import load_dotenv

    load_dotenv()
except Exception:  # skip .env if python-dotenv is missing or fails
    pass
st.set_page_config(page_title="Study Day Planner", page_icon="📚")
st.title("📚 Study Day Planner")
st.caption("Tasks, weather and books — with approval before deleting anything.")

if not os.environ.get("GROQ_API_KEY"):
    st.error("GROQ_API_KEY is not set. Copy .env.example to .env and add your key.")
    st.stop()

from agent import answer, build_graph, chat, last_text


@st.cache_resource
def get_graph():
    return build_graph()


graph = get_graph()
if "thread_id" not in st.session_state:
    st.session_state.thread_id = str(uuid.uuid4())
    st.session_state.history = []   # [(role, text)]
    st.session_state.pending = None  # approval request, if any
config = {"configurable": {"thread_id": st.session_state.thread_id}}

for role, text in st.session_state.history:
    st.chat_message(role).write(text)


def finish(result, pending):
    st.session_state.pending = pending
    if not pending:
        st.session_state.history.append(("assistant", last_text(result)))


if st.session_state.pending:
    st.warning(st.session_state.pending["question"])
    c1, c2 = st.columns(2)
    decision = None
    if c1.button("✅ Yes, delete"):
        decision = True
    if c2.button("❌ No, keep it"):
        decision = False
    if decision is not None:
        finish(*answer(graph, decision, config))
        st.rerun()
else:
    if prompt := st.chat_input("Ask me about tasks, weather or books…"):
        st.session_state.history.append(("user", prompt))
        finish(*chat(graph, prompt, config))
        st.rerun()